# Eksperimen — Heart Failure Prediction

**Nama:** handokobeni  
**Dataset:** Heart Failure Prediction (target: `HeartDisease`, klasifikasi biner)  
**Tujuan:** eksperimen manual (data loading, EDA, preprocessing) sebelum diotomasi pada `automate_handokobeni.py`.

## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split

## 2. Data Loading

Memuat dataset mentah dan meninjau strukturnya.

In [ ]:
df = pd.read_csv("../heart_raw.csv")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print("Jumlah missing value per kolom:")
df.isnull().sum()

## 3. Exploratory Data Analysis (EDA)

Menganalisis distribusi, outlier, dan hubungan antar fitur.

### 3.1 Distribusi kelas target

In [ ]:
sns.countplot(x="HeartDisease", data=df)
plt.title("Distribusi HeartDisease")
plt.show()

### 3.2 Distribusi fitur numerik

In [ ]:
df.select_dtypes(include="number").hist(figsize=(12, 8))
plt.tight_layout()
plt.show()

### 3.3 Deteksi outlier (contoh: Cholesterol)

In [ ]:
sns.boxplot(x=df["Cholesterol"])
plt.title("Boxplot Cholesterol")
plt.show()

### 3.4 Matriks korelasi fitur numerik

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.select_dtypes(include="number").corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()

**Insight EDA:**
- Dataset relatif seimbang antara kelas 0 (tidak) dan 1 (berisiko penyakit jantung).
- Terdapat indikasi outlier/nilai 0 tidak wajar pada `Cholesterol` dan `RestingBP`.
- Fitur seperti `Oldpeak`, `MaxHR`, dan `ST_Slope` berkorelasi cukup kuat dengan target.
- Ada campuran fitur numerik & kategorikal → butuh scaling + encoding.

## 4. Data Preprocessing

Membersihkan & menyiapkan data: imputasi, scaling numerik, encoding kategorikal, lalu split.

In [ ]:
X = df.drop(columns=["HeartDisease"])
y = df["HeartDisease"]

numeric = X.select_dtypes(include="number").columns.tolist()
categorical = X.select_dtypes(include="object").columns.tolist()
print("Numerik:", numeric)
print("Kategorikal:", categorical)

In [ ]:
# Imputasi + scaling fitur numerik
X[numeric] = SimpleImputer(strategy="median").fit_transform(X[numeric])
X[numeric] = StandardScaler().fit_transform(X[numeric])

In [ ]:
# Encoding fitur kategorikal (one-hot)
enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoded = enc.fit_transform(X[categorical])
X_enc = pd.concat(
    [X[numeric].reset_index(drop=True),
     pd.DataFrame(encoded, columns=enc.get_feature_names_out(categorical))],
    axis=1,
)
X_enc.head()

In [ ]:
# Split data latih & uji
X_train, X_test, y_train, y_test = train_test_split(
    X_enc, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape, " Test:", X_test.shape)

In [ ]:
# Simpan dataset hasil preprocessing
X_enc.assign(HeartDisease=y).to_csv("heart_preprocessing.csv", index=False)
print("Tersimpan: heart_preprocessing.csv", X_enc.shape)

## 5. Kesimpulan

Seluruh tahap eksperimen (data loading, EDA, preprocessing) di atas telah dikonversi menjadi fungsi otomatis pada **`automate_handokobeni.py`**, yang dijalankan otomatis via **GitHub Actions** setiap ada perubahan.